In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import center_of_mass
import tensorflow as tf

from GT_func_file_Model import ResNet_model_paper
from GT_func_file_NPC import *

## Load NPC data

The datafile with nuclear pore complexes appears to be corrupted from the start but loads 104,614 frames correctly. Therefore, we only keep the last 50,000 where the corruption should not present. <br>
Additionally, we take 5,000 frames to allow for reconstruction without the need of drift correction.

In [ ]:
#Load the 5k frames from the nuclear pore complexes
path = "NPC_data.tif"   #Set the path to the NPC dataset

frames_5k, norms_5k = Load_NPC_stack(path)
frames_5k.shape

## The DAMN reconstruction

Tensorflow may print warnings.

In [ ]:
#Load the model
model_ResNet = ResNet_model_paper(channels = 64, num_blocks_array = [3,3,3,3], 
                                  kernel_sizes=[5,7,9,11], LeakyReLU_slope=0.1, 
                                  padding="same", interpolation="bilinear", kernel_initializer="he_uniform")

#Compile and load weights
_ = model_ResNet(tf.random.normal([1, 50, 50, 1]))
model_ResNet.load_weights("DAMN_upsampling_model.keras")

In [ ]:
#Reconstruct all data using the model. Takes approximately 5 min on our computational resources.

evaluation_batch = 32   #Set the evaluation batch_size depending on your available GPU memory

DAMN_predicted = np.squeeze(model_ResNet.predict(np.expand_dims(frames_5k, axis=-1), batch_size=evaluation_batch, verbose=1))

DAMN_reconstructed = np.mean(DAMN_predicted * norms_5k[:,None,None], axis=0)
DAMN_reconstructed.shape

## Vizualize a single NPC

In [ ]:
#Choose the targeted single NPC and load the Deep-STORM reconstruction of the same NPC
single_DAMN_NPC = DAMN_reconstructed[203:228,375:]
single_DS_NPC = np.load("DS_single_NPC.npy")

In [ ]:
#Vizualize both the reconstruction
plt.figure(figsize=(15,5))
plt.subplot(131)
plt.matshow(single_DS_NPC, fignum=False)
plt.title("NPC reconstruction (DS)")

plt.subplot(132)
plt.matshow(single_DAMN_NPC, fignum=False)
plt.title("NPC reconstruction (DAMN)")

plt.subplot(133)
plt.matshow(single_DAMN_NPC, fignum=False)
plt.title("NPC reconstruction (DAMN) with distances")

#And add the targeted distances to the DAMN NPC reconstruction
x2 = 8.25
y2 = 7.25
plt.hlines(y=11.5, xmin=x2, xmax=x2+8.56, color="red", label="107nm")
plt.vlines(x=12.25, ymin=y2, ymax=y2+8.56, color="red")
plt.plot([8,9], [10.75,14], color="blue", label="42nm")
plt.plot([10.75,14], [8,8.75], color="blue")

plt.legend()

### Find centers of mass

In [ ]:
#Restricting the individual emitters to center their masses
positions = np.array([
    [13,16,8,11], [10,13,8,11], [7,10,10,13], [7,10,13,16], [9,12,15,18]
])

In [ ]:
#Compute the centers of masss based on the restrictions
DAMN_COMs = np.zeros([positions.shape[0],2])
DS_COMs = np.zeros([positions.shape[0],2])

for i in range(positions.shape[0]):
    DAMN_COMs[i] = center_of_mass(single_DAMN_NPC[positions[i,0]:positions[i,1], positions[i,2]:positions[i,3]]) + np.array([positions[i,0], positions[i,2]])
    DS_COMs[i] = center_of_mass(single_DS_NPC[positions[i,0]:positions[i,1], positions[i,2]:positions[i,3]]) + np.array([positions[i,0], positions[i,2]])

In [ ]:
#Vizualize the images again, with the centers of mass
plt.figure(figsize=(10,5))
plt.subplot(121)
plt.matshow(single_DS_NPC, fignum=False)
for i in range(DS_COMs.shape[0]):
    plt.scatter(DS_COMs[i,1], DS_COMs[i,0], marker="+", color="red", s=50)
plt.plot()
plt.title("Deep-STORM")

plt.subplot(122)
plt.matshow(single_DAMN_NPC, fignum=False)
for i in range(DAMN_COMs.shape[0]):
    plt.scatter(DAMN_COMs[i,1], DAMN_COMs[i,0], marker="+", color="red", s=50)
plt.title("DAMN")

### Estimate distances

In [ ]:
#Estimate the emitter distances based on the centers of mass
DAMN_distances = np.zeros([positions.shape[0]-1])
DS_distances = np.zeros([positions.shape[0]-1])

for i in range(DAMN_distances.shape[0]):
    DAMN_distances[i] = np.sqrt(np.sum((DAMN_COMs[i] - DAMN_COMs[i+1])**2))
    DS_distances[i] = np.sqrt(np.sum((DS_COMs[i] - DS_COMs[i+1])**2))

DAMN_ring_diameter = np.sqrt(np.sum((DAMN_COMs[0] - DAMN_COMs[-1])**2))
DS_ring_diameter = np.sqrt(np.sum((DS_COMs[0] - DS_COMs[-1])**2))

In [ ]:
#Recalculate the estimated distances from reconstructed pixels to real nm
pixel_to_nm = 12.5    #Based on data information

#Nearest-neighbor emitter distances
DAMN_distances_nm = DAMN_distances * pixel_to_nm
DS_distances_nm = DS_distances * pixel_to_nm

#Diameter
DAMN_ring_diameter_nm = DAMN_ring_diameter * pixel_to_nm
DS_ring_diameter_nm = DS_ring_diameter * pixel_to_nm

#Average the five nearest-neighbor emitter distances
DAMN_aver_distances_nm = np.mean(DAMN_distances_nm)
DS_aver_distances_nm = np.mean(DS_distances_nm)

In [ ]:
print("Average inter-emitter distance for DAMN:", np.round(DAMN_aver_distances_nm, 1), "nm")
print("Average inter-emitter distance for DS:  ", np.round(DS_aver_distances_nm, 1), "nm")

In [ ]:
print("Ring diameter estimate for DAMN:", np.round(DAMN_ring_diameter_nm, 1), "nm")
print("Ring diameter estimate for DS:  ", np.round(DS_ring_diameter_nm, 1), "nm")